# Swiss parliamentary source import

This notebook recreates the country-neutral SQLite research database exclusively from the committed schema and verified local source manifest. It performs no network requests and no political classification.

## Locate the repository and load the transparent import helper

The search walks upward from the kernel's current directory, so execution works from the repository root or the `notebooks` directory without hard-coded machine paths.

In [1]:
from pathlib import Path
import json
import os
import sqlite3
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
repository_root = next(
    (path for path in candidates if (path / 'database/schema.sql').is_file() and (path / '.agents/CONTEXT.md').is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError('Repository root not found above the current working directory')
sys.path.insert(0, str(repository_root / 'src'))

from politiks.importer import query_rows, recreate_database

database_path = repository_root / 'database/parliament.sqlite'
manifest_relative = os.environ.get(
    'POLITIKS_SOURCE_MANIFEST', 'source/manifests/fixture.jsonl'
)
manifest_path = repository_root / manifest_relative
print('Repository root: located')
print(f'Local manifest: {manifest_path.relative_to(repository_root).as_posix()}')
print(f'Generated database: {database_path.relative_to(repository_root).as_posix()}')

Repository root: located
Local manifest: source/manifests/full_swiss_2026-07-14.jsonl
Generated database: database/parliament.sqlite


## Recreate and import the database

The helper deletes only the generated SQLite file, applies `database/schema.sql`, verifies every source byte count and SHA256 against the manifest, and imports all supported JSON shapes in one transaction. Non-JSON documentation is still registered for provenance.

In [2]:
report = recreate_database(
    repository_root, manifest_path=manifest_path, database_path=database_path
)
print(json.dumps({
    'snapshot': report.snapshot_name,
    'source_files': report.source_files,
    'source_records': report.source_records,
    'normalized_files': report.normalized_files,
    'registered_only': len(report.skipped_files),
}, ensure_ascii=False, indent=2))

{
  "snapshot": "full_swiss_2026-07-14",
  "source_files": 362,
  "source_records": 34646,
  "normalized_files": 360,
  "registered_only": 2
}


## Inspect bounded logical row counts

These counts make repeated imports comparable and show the boundary between provenance records, parliamentary entities, and individual voting choices.

In [3]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(
    [{'table': table, 'rows': rows} for table, rows in report.table_counts.items()]
).style.hide(axis='index'))
display(pd.DataFrame(
    [{'recorded_choice': choice, 'rows': rows} for choice, rows in report.choice_counts.items()]
).style.hide(axis='index'))

table,rows
source_file,362
source_record,34646
country,1
legislature,1
chamber,3
legislative_period,16
parliamentary_session,22
subdivision,26
committee,341
political_party,83


recorded_choice,rows
abstain,64485
excused,52981
no,1310702
not_participating,173534
other,339
presiding,20096
yes,2239453


## Report coverage and unresolved links explicitly

Coverage limitations are reported rather than guessed away. The fixture's sampled vote payloads omit a chamber field, while the full session spreadsheets identify both chambers explicitly. Membership gaps remain counted, and inferred profile-derived intervals stay distinguishable from explicit dated evidence.

In [4]:
display(pd.DataFrame(report.chamber_year_counts).style.hide(axis='index'))
limitations = {
    'voting events with unresolved chamber': report.unresolved_event_chambers,
    'voting events without an affair': report.unlinked_event_matters,
    'choices without date-valid party membership': report.choices_without_dated_party,
    'choices without date-valid faction membership': report.choices_without_dated_faction,
}
print(json.dumps(limitations, ensure_ascii=False, indent=2))

chamber,year,voting_events
Nationalrat,2011,240
Nationalrat,2012,1211
Nationalrat,2013,1041
Nationalrat,2014,1040
Nationalrat,2015,1174
Nationalrat,2016,1218
Nationalrat,2017,1123
Nationalrat,2018,1036
Nationalrat,2019,1106
Nationalrat,2020,1597


{
  "voting events with unresolved chamber": 0,
  "voting events without an affair": 17,
  "choices without date-valid party membership": 39445,
  "choices without date-valid faction membership": 120
}


## Verify a representative evidence join

This query follows one choice through its voting event and affair to the person and the party/faction interval covering the vote date. The `is_inferred` and `evidence_basis` columns prevent current-profile associations from masquerading as explicit historical membership.

In [5]:
representative_join = query_rows(database_path, '''
SELECT ve.source_identifier AS vote_id, ve.occurred_at, pm.formatted_identifier AS affair_id,
       pm.title, p.display_name, vc.raw_decision, vc.normalized_choice,
       pp.abbreviation AS party, pf.abbreviation AS faction,
       ppm.is_inferred AS party_is_inferred, ppm.evidence_basis AS party_basis
FROM voting_choice vc
JOIN voting_event ve ON ve.id = vc.voting_event_id
JOIN parliamentary_matter pm ON pm.id = ve.matter_id
JOIN person p ON p.id = vc.person_id
JOIN person_party_membership ppm ON ppm.person_id = p.id
 AND (ppm.date_from IS NULL OR ppm.date_from <= substr(ve.occurred_at, 1, 10))
 AND (ppm.date_to IS NULL OR ppm.date_to >= substr(ve.occurred_at, 1, 10))
JOIN political_party pp ON pp.id = ppm.party_id
JOIN person_faction_membership pfm ON pfm.person_id = p.id
 AND (pfm.date_from IS NULL OR pfm.date_from <= substr(ve.occurred_at, 1, 10))
 AND (pfm.date_to IS NULL OR pfm.date_to >= substr(ve.occurred_at, 1, 10))
JOIN parliamentary_faction pf ON pf.id = pfm.faction_id
WHERE p.display_name = 'Thomas Aeschi'
ORDER BY ve.occurred_at DESC LIMIT 3
''')
display(pd.DataFrame(representative_join).style.hide(axis='index'))

vote_id,occurred_at,affair_id,title,display_name,raw_decision,normalized_choice,party,faction,party_is_inferred,party_basis
NR:32796,2026-06-19,20.0406,"Unternehmerinnen und Unternehmer, welche Beiträge an die Arbeitslosenversicherung bezahlen, sollen auch gegen Arbeitslosigkeit versichert sein",Thomas Aeschi,nein,no,SVP,V,0,explicit_historic_membership_interval
NR:32796,2026-06-19,20.0406,"Unternehmerinnen und Unternehmer, welche Beiträge an die Arbeitslosenversicherung bezahlen, sollen auch gegen Arbeitslosigkeit versichert sein",Thomas Aeschi,nein,no,SVP,V,0,explicit_historic_membership_interval
NR:32785,2026-06-19,20.0504,Folter als eigener Straftatbestand im Schweizer Strafrecht,Thomas Aeschi,nein,no,SVP,V,0,explicit_historic_membership_interval


## Enforce final integrity and identifier checks

The final cell fails execution on foreign-key violations, duplicate stable identifiers, missing source files, or an empty representative join. Passing it makes the notebook an executable import contract rather than a narrative-only artifact.

In [6]:
checks = query_rows(database_path, '''
SELECT
  (SELECT COUNT(*) FROM pragma_foreign_key_check) AS foreign_key_violations,
  (SELECT COUNT(*) FROM (
     SELECT source_system, namespace, identifier FROM person_identifier
     GROUP BY source_system, namespace, identifier HAVING COUNT(*) > 1
   )) AS duplicate_person_identifiers,
  (SELECT COUNT(*) FROM (
     SELECT source_system, source_identifier FROM voting_event
     GROUP BY source_system, source_identifier HAVING COUNT(*) > 1
   )) AS duplicate_voting_identifiers,
  (SELECT COUNT(*) FROM source_file) AS source_files
''')[0]
assert checks['foreign_key_violations'] == 0
assert checks['duplicate_person_identifiers'] == 0
assert checks['duplicate_voting_identifiers'] == 0
assert checks['source_files'] == report.source_files
assert representative_join
print(json.dumps(checks, ensure_ascii=False, indent=2))
print('Import and integrity checks passed.')

{
  "foreign_key_violations": 0,
  "duplicate_person_identifiers": 0,
  "duplicate_voting_identifiers": 0,
  "source_files": 362
}
Import and integrity checks passed.
